# 3.对话链

知识点讲解：对话链与消息历史管理（ConversationChain 的现代替代方案）

1. 对话链要解决的核心问题
   - LLM 本身无状态，每次调用之间不会记住之前内容
   - 多轮对话需将历史消息一并送入模型，才能实现"记忆"效果
   - 对话链职责：自动完成历史消息的读取、注入与写回

2. 从 ConversationChain 到 RunnableWithMessageHistory
   - 0.x 的 ConversationChain 内置 ConversationBufferMemory，历史读写对用户不可见、隐式存储
   - 隐式存储问题：难以定制、难以支持多用户会话隔离、难以持久化
   - 1.x 的 RunnableWithMessageHistory 采用显式设计，由开发者提供历史存储函数
   - 课程示例：ConversationChain(llm=ChatOpenAI(model="gpt-4o"))，invoke 返回 dict 含 input / history / response 三个键，history 由内部 Memory 自动维护（旧写法，已不推荐）

3. RunnableWithMessageHistory 的三个关键参数
   - runnable：被包装的底层链，必须包含 MessagesPlaceholder 用于接收历史
   - get_session_history：回调函数，根据 session_id 返回 BaseChatMessageHistory
   - input_messages_key / history_messages_key：指定输入与历史在 dict 中的键名

4. MessagesPlaceholder 的作用
   - 在提示词模板中预留一个"消息列表插槽"
   - invoke 时由 RunnableWithMessageHistory 自动填入历史消息列表
   - 位置关键：通常放在 system 之后、human 之前，保证对话顺序正确

5. session_id 与会话隔离
   - 通过 config={"configurable": {"session_id": "..."}} 指定当前会话
   - 不同 session_id 对应不同历史记录，实现多用户 / 多会话隔离
   - 生产环境中 session_id 通常来自用户 ID 或会话 UUID

6. BaseChatMessageHistory 的常见实现
   - InMemoryChatMessageHistory：内存存储，进程重启后丢失，适合演示与测试
   - FileChatMessageHistory：文件存储，可跨进程持久化
   - RedisChatMessageHistory / SQLChatMessageHistory：生产级持久化方案

7. 完整的数据流转过程
   - 调用前：从 store 读取 session_id 对应的历史 → 填入 history 插槽
   - 执行中：{"input": ..., "history": [...]} → prompt → llm → parser → str
   - 调用后：将本轮 HumanMessage 与 AIMessage 自动追加写回历史


In [ ]:
import dotenv
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
llm = ChatOpenAI(model="deepseek-v4-pro")

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是DeepSeek开发的聊天机器人，请根据对话上下文回复用户问题"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()
store: dict[str, BaseChatMessageHistory] = {}

In [ ]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """根据会话 id 获取消息历史，替代 Memory 组件的隐式存储
    这是 RunnableWithMessageHistory 要求的工厂函数（factory function）。
    参数：
        session_id: str - 会话唯一标识，由 config.configurable.session_id 传入
    返回：
        BaseChatMessageHistory - 该会话对应的历史存储对象
        该对象需实现 messages 属性（读）和 add_message 方法（写）
    调用时机：
        每次 invoke/stream 时被 RunnableWithMessageHistory 自动调用两次
        - 执行前：读取历史填入 MessagesPlaceholder
        - 执行后：写回本轮的 Human 与 AI 消息
    实现要点：
        必须保证同一 session_id 返回同一个历史对象实例，否则历史会丢失
    """
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
config = {"configurable": {"session_id": "muxiaoke"}}
content = chain_with_history.invoke(
    {"input": "你好，我是慕小课，我喜欢打篮球还有游泳，你喜欢什么运动呢？"},
    config=config,
)
print(content)
content = chain_with_history.invoke(
    {"input": "根据上下文信息，请统计一下我的运动爱好有什么?"},
    config=config,
)
print(content)